In [1]:
import cv2
import os

# --- Configuration ---
IMG_DIR = "Annotated_Better_Again"
OUT_DIR = "Annotated_Better_Again/labels"
os.makedirs(OUT_DIR, exist_ok=True)
MIN_BOX_SIZE = 5  # Minimum pixel size to count as a valid box (prevents ghost clicks)

scale = .25
display_img = None
clone = None 
drawing = False
ix, iy = -1, -1
boxes = []

def draw_box(event, x, y, flags, param):
    global ix, iy, drawing, boxes, display_img, clone
    
    # Start drawing
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        ix, iy = x, y

    # Visualization while dragging (Optional polish)
    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        temp_img = display_img.copy()
        cv2.rectangle(temp_img, (ix, iy), (x, y), (0, 255, 0), 2)
        cv2.imshow("image", temp_img)

    # Stop drawing and save box
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        
        # Calculate coordinates
        x_min, y_min = min(ix, x), min(iy, y)
        x_max, y_max = max(ix, x), max(iy, y)
        
        # 1. FILTER: Ignore tiny boxes (Ghost clicks)
        if (x_max - x_min) < MIN_BOX_SIZE or (y_max - y_min) < MIN_BOX_SIZE:
            print("Ignored tiny box (likely a click without drag)")
            cv2.imshow("image", display_img) # Refresh to clear drag visual
            return

        # Scale back to original size for saving
        x_min_orig = int(x_min / scale)
        y_min_orig = int(y_min / scale)
        x_max_orig = int(x_max / scale)
        y_max_orig = int(y_max / scale)

        boxes.append((x_min_orig, y_min_orig, x_max_orig, y_max_orig))

        # Draw permanently on display image
        cv2.rectangle(display_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        update_display_text() # Update the counter
        cv2.imshow("image", display_img)

def update_display_text():
    """Refreshes the image and redraws all boxes + counter"""
    global display_img
    # Reset display to clean clone
    display_img = clone.copy()
    
    # Redraw all existing boxes
    for (x1, y1, x2, y2) in boxes:
        # Scale orig coords back down to display coords
        d_x1, d_y1 = int(x1 * scale), int(y1 * scale)
        d_x2, d_y2 = int(x2 * scale), int(y2 * scale)
        cv2.rectangle(display_img, (d_x1, d_y1), (d_x2, d_y2), (0, 255, 0), 2)
    
    # 2. COUNTER: Draw the box count on screen
    cv2.putText(display_img, f"Count: {len(boxes)}", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

def save_yolo_format(img_name, boxes, img_w, img_h):
    label_path = os.path.join(OUT_DIR, os.path.splitext(img_name)[0] + ".txt")
    with open(label_path, "w") as f:
        for (x1, y1, x2, y2) in boxes:
            # Avoid division by zero if something went wrong
            if img_w == 0 or img_h == 0: continue
            
            x_center = ((x1 + x2) / 2) / img_w
            y_center = ((y1 + y2) / 2) / img_h
            w = (x2 - x1) / img_w
            h = (y2 - y1) / img_h
            f.write(f"0 {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")
    print(f"Saved {len(boxes)} labels for {img_name}")

# --- Main Loop ---
for img_name in os.listdir(IMG_DIR):
    if not img_name.lower().endswith((".png", ".jpg", ".jpeg")):
        continue
        
    img_path = os.path.join(IMG_DIR, img_name)
    orig_img = cv2.imread(img_path)
    if orig_img is None: continue

    # Resize for display
    display_img = cv2.resize(orig_img, (int(orig_img.shape[1] * scale),
                                      int(orig_img.shape[0] * scale)))
    clone = display_img.copy()
    boxes = []
    
    # Initial text update
    update_display_text()

    cv2.namedWindow("image")
    cv2.setMouseCallback("image", draw_box)

    print(f"Annotating: {img_name}")
    print("Controls: [S] Save & Next | [Z] Undo Last Box | [R] Reset All | [ESC] Quit")

    while True:
        cv2.imshow("image", display_img)
        key = cv2.waitKey(1) & 0xFF

        if key == ord("s"):  # save and move on
            save_yolo_format(img_name, boxes, orig_img.shape[1], orig_img.shape[0])
            break
            
        elif key == ord("z"): # 3. UNDO feature
            if len(boxes) > 0:
                boxes.pop()
                update_display_text()
                print(f"Undo. Count: {len(boxes)}")
                
        elif key == ord("r"):  # reset boxes
            boxes = []
            update_display_text()
            print("Reset boxes")
            
        elif key == 27:  # ESC to quit
            cv2.destroyAllWindows()
            exit()

cv2.destroyAllWindows()

Annotating: 101142060_00003.jpg
Controls: [S] Save & Next | [Z] Undo Last Box | [R] Reset All | [ESC] Quit
Saved 37 labels for 101142060_00003.jpg
Annotating: 101142060_00004.jpg
Controls: [S] Save & Next | [Z] Undo Last Box | [R] Reset All | [ESC] Quit
Saved 37 labels for 101142060_00004.jpg
Annotating: 101142060_00005.jpg
Controls: [S] Save & Next | [Z] Undo Last Box | [R] Reset All | [ESC] Quit
Saved 37 labels for 101142060_00005.jpg
Annotating: 101142060_00006.jpg
Controls: [S] Save & Next | [Z] Undo Last Box | [R] Reset All | [ESC] Quit
Saved 37 labels for 101142060_00006.jpg
Annotating: 101142060_00007.jpg
Controls: [S] Save & Next | [Z] Undo Last Box | [R] Reset All | [ESC] Quit
Ignored tiny box (likely a click without drag)
Saved 37 labels for 101142060_00007.jpg
Annotating: 101142060_00008.jpg
Controls: [S] Save & Next | [Z] Undo Last Box | [R] Reset All | [ESC] Quit
Saved 37 labels for 101142060_00008.jpg
Annotating: 101142060_00009.jpg
Controls: [S] Save & Next | [Z] Undo La